# Meteorite Classification - 9th Place Solution (K-Fold + Focal Loss + TTA)
Based on the Kaggle Classify Leaves 9th place solution:
1. 5-Fold Stratified Cross Validation
2. Focal Loss for handling severe class imbalance
3. Heavy Data Augmentation & TTA (Test Time Augmentation)
4. Model Ensembling from K-Fold


In [ ]:
import os
import random
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset

import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from tqdm.auto import tqdm

import timm
from dataset import StoneDataset, resolve_train_paths, resolve_test_paths

# Configuration
CFG = {
    'seed': 42,
    'data_root': './dataset', # Adjust path if needed
    'n_splits': 5,            # 5-fold CV
    'model_name': 'tf_efficientnetv2_s',
    'image_size': 224,
    'batch_size': 32,
    'epochs': 10,
    'lr': 3e-4,
    'min_lr': 1e-6,
    'weight_decay': 1e-4,
    'num_workers': 4,
    'tta_views': 4,           # TTA flips
    'num_classes': 2,
    'focal_gamma': 2.0,
    'use_focal': True,
    'device': torch.device('cuda' if torch.cuda.is_available() else 'cpu')
}

DATA_ROOT_CANDIDATES = [
    './dataset',
    '/data/meteorite-identification',
    './data',
]

def detect_data_root(candidates):
    for root in candidates:
        try:
            resolve_train_paths(root)
            resolve_test_paths(root)
            return root
        except Exception:
            continue
    raise FileNotFoundError(
        'No valid dataset root found. Expected train_images/train_labels.csv/test_images/sample_submission.csv'
    )

CFG['data_root'] = detect_data_root(DATA_ROOT_CANDIDATES)

print(f"Using Device: {CFG['device']}")
print(f"Data Root: {CFG['data_root']}")

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(CFG['seed'])

/opt/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using Device: cuda
Data Root: /data/meteorite-identification


In [2]:
# Data loading and Albumentations
import cv2

class AlbumentationsTransform:
    def __init__(self, aug):
        self.aug = aug
    def __call__(self, image):
        # PIL to numpy
        image_np = np.array(image)
        augmented = self.aug(image=image_np)
        return augmented['image']

train_transform = AlbumentationsTransform(
    A.Compose([
        A.Resize(CFG['image_size'], CFG['image_size']),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=30, p=0.5),
        # A.RandomResizedCrop(CFG['image_size'], CFG['image_size'], p=0.8),
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05, p=0.7),
        A.CLAHE(p=0.3),
        A.CoarseDropout(max_holes=8, max_height=16, max_width=16, p=0.3),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])
)

eval_transform = AlbumentationsTransform(
    A.Compose([
        A.Resize(CFG['image_size'], CFG['image_size']),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])
)

# Focal Loss Implementation
class FocalLossStable(nn.Module):
    """Numerically stable focal loss for multi-class/binary classification"""
    def __init__(self, gamma=2.0, alpha=None, reduction='mean'):
        super().__init__()
        self.gamma = gamma
        self.reduction = reduction
        self.alpha = None if alpha is None else torch.as_tensor(alpha, dtype=torch.float32)

    def forward(self, logits, targets):
        targets = targets.view(-1).long()
        log_probs = F.log_softmax(logits, dim=1)
        log_pt = log_probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        pt = log_pt.exp()
        
        focal_factor = (1.0 - pt).clamp(min=0.0).pow(self.gamma)
        loss = -focal_factor * log_pt
        
        if self.alpha is not None:
            alpha = self.alpha.to(device=logits.device, dtype=logits.dtype).view(-1)
            loss = alpha.gather(0, targets) * loss
            
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        return loss

/opt/venv/lib/python3.12/site-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/tmp/ipykernel_103312/461268601.py:22: UserWarning: Argument(s) 'max_holes, max_height, max_width' are not valid for transform CoarseDropout
  A.CoarseDropout(max_holes=8, max_height=16, max_width=16, p=0.3),


In [3]:
def load_local_weights_flexible(model, weight_path):
    if not os.path.exists(weight_path):
        raise FileNotFoundError(f'Local weight file not found: {weight_path}')

    if weight_path.endswith('.safetensors'):
        from safetensors.torch import load_file as safe_load_file
        raw_state = safe_load_file(weight_path)
    else:
        ckpt = torch.load(weight_path, map_location='cpu')
        if isinstance(ckpt, dict) and 'state_dict' in ckpt:
            raw_state = ckpt['state_dict']
        else:
            raw_state = ckpt

    model_state = model.state_dict()
    filtered_state = {}
    skipped_keys = []

    for k, v in raw_state.items():
        key = k[7:] if k.startswith('module.') else k
        if key in model_state and model_state[key].shape == v.shape:
            filtered_state[key] = v
        else:
            skipped_keys.append(key)

    missing, unexpected = model.load_state_dict(filtered_state, strict=False)
    return {
        'loaded_tensors': len(filtered_state),
        'skipped_tensors': len(skipped_keys),
        'missing_keys': len(missing),
        'unexpected_keys': len(unexpected),
    }


def build_model(model_name, num_classes=2, pretrained=True):
    # Determine the local weight path based on the chosen model, or hardcode here.
    # Defaulting to './weights/model.safetensors' as in original script, but can be updated.
    weight_path = './weights/model.safetensors' 
    use_local_pretrained = os.path.exists(weight_path)

    if use_local_pretrained:
        print(f"Loading local weights from {weight_path}")
        model = timm.create_model(
            model_name,
            pretrained=False,
            num_classes=num_classes,
            drop_rate=0.5
        )
        try:
            stats = load_local_weights_flexible(model, weight_path)
            print(f"Local weights load stats: {stats}")
        except Exception as e:
            print(f"Failed to load local weights: {e}, falling back...")
            model = timm.create_model(
                model_name,
                pretrained=pretrained,
                num_classes=num_classes,
                drop_rate=0.5
            )
    else:
        print(f"No local weights found at {weight_path}, using timm pretrained={pretrained}")
        model = timm.create_model(
            model_name,
            pretrained=pretrained,
            num_classes=num_classes,
            drop_rate=0.5
        )
    return model

def tta_forward(model, images, views=4):
    """Test-time augmentation with flips"""
    model.eval()
    with torch.no_grad():
        logits_list = []
        logits_list.append(model(images))
        if views >= 2:
            logits_list.append(model(torch.flip(images, dims=[3]))) # H-flip
        if views >= 3:
            logits_list.append(model(torch.flip(images, dims=[2]))) # V-flip
        if views >= 4:
            logits_list.append(model(torch.flip(images, dims=[2, 3]))) # Both
        
        prob_stack = [torch.softmax(lg, dim=1) for lg in logits_list]
        return torch.stack(prob_stack, dim=0).mean(dim=0)

def compute_class_weights(labels_np):
    counts = np.bincount(labels_np)
    smoothing = 1e-6
    prior = (counts + smoothing) / (counts.sum() + smoothing * len(counts))
    weights = 1.0 / prior
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32)

def evaluate(model, loader, device, tta_views=1):
    model.eval()
    y_true, y_probs = [], []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            if tta_views > 1:
                probs = tta_forward(model, images, views=tta_views)
            else:
                logits = model(images)
                probs = torch.softmax(logits, dim=1)
            
            y_probs.extend(probs[:, 1].cpu().numpy())
            y_true.extend(labels.cpu().numpy())
    
    y_pred = (np.array(y_probs) >= 0.5).astype(int)
    f1 = f1_score(y_true, y_pred, average='binary', zero_division=0)
    return f1, np.array(y_probs)

In [4]:
def run_kfold():
    os.makedirs('weights', exist_ok=True)
    full_dataset = StoneDataset(root=CFG['data_root'], split='train', transforms=None)
    labels = np.array(full_dataset.labels).astype(int)
    
    skf = StratifiedKFold(n_splits=CFG['n_splits'], shuffle=True, random_state=CFG['seed'])
    
    oof_probs = np.zeros(len(full_dataset))
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(labels)), labels)):
        print(f"========== Fold {fold+1}/{CFG['n_splits']} ==========")
        
        train_ds_raw = StoneDataset(root=CFG['data_root'], split='train', transforms=train_transform)
        val_ds_raw = StoneDataset(root=CFG['data_root'], split='train', transforms=eval_transform)
        
        train_dataset = Subset(train_ds_raw, train_idx)
        val_dataset = Subset(val_ds_raw, val_idx)
        
        train_loader = DataLoader(train_dataset, batch_size=CFG['batch_size'], shuffle=True, 
                                  num_workers=CFG['num_workers'], pin_memory=True)
        val_loader = DataLoader(val_dataset, batch_size=CFG['batch_size'], shuffle=False, 
                                num_workers=CFG['num_workers'], pin_memory=True)
        
        model = build_model(CFG['model_name']).to(CFG['device'])
        
        y_train = labels[train_idx]
        class_weights = compute_class_weights(y_train)
        if CFG['use_focal']:
            criterion = FocalLossStable(gamma=CFG['focal_gamma'], alpha=class_weights).to(CFG['device'])
        else:
            criterion = nn.CrossEntropyLoss(weight=class_weights).to(CFG['device'])
            
        optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG['epochs'], eta_min=CFG['min_lr'])
        
        best_f1 = 0
        for epoch in range(CFG['epochs']):
            model.train()
            running_loss = 0.0
            
            pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CFG['epochs']}", leave=False)
            for images, targets in pbar:
                images, targets = images.to(CFG['device']), targets.to(CFG['device'])
                
                optimizer.zero_grad()
                logits = model(images)
                loss = criterion(logits, targets)
                
                loss.backward()
                optimizer.step()
                running_loss += loss.item() * targets.size(0)
            
            scheduler.step()
            
            val_f1, _ = evaluate(model, val_loader, CFG['device'], tta_views=1) # normal validation without TTA
            print(f"Epoch {epoch+1}: Loss = {running_loss/len(train_dataset):.4f}, Val F1 = {val_f1:.4f}")
            
            if val_f1 > best_f1:
                best_f1 = val_f1
                torch.save(model.state_dict(), f"weights/fold{fold}_{CFG['model_name']}_best.pt")
                print(f"--> Saved best model for fold {fold}")
                
        # Generate OOF with TTA and best model
        model.load_state_dict(torch.load(f"weights/fold{fold}_{CFG['model_name']}_best.pt", map_location=CFG['device'], weights_only=True))
        _, val_probs = evaluate(model, val_loader, CFG['device'], tta_views=CFG['tta_views'])
        oof_probs[val_idx] = val_probs
        
    y_pred_oof = (oof_probs >= 0.5).astype(int)
    overall_f1 = f1_score(labels, y_pred_oof, average='binary', zero_division=0)
    print(f"Overall OOF F1-Score: {overall_f1:.4f}")
    
    return oof_probs

# Run K-Fold training
oof_probs = run_kfold()

========== Fold 1/5 ==========
Loading local weights from ./weights/model.safetensors
Local weights load stats: {'loaded_tensors': 780, 'skipped_tensors': 2, 'missing_keys': 2, 'unexpected_keys': 0}


Epoch 1: Loss = 1.2419, Val F1 = 0.9746
--> Saved best model for fold 0


Epoch 2: Loss = 0.1613, Val F1 = 0.9740


Epoch 3: Loss = 0.1010, Val F1 = 0.9842
--> Saved best model for fold 0


Epoch 4: Loss = 0.0789, Val F1 = 0.9802


...

Epoch 4: Loss = 0.0980, Val F1 = 0.9813
--> Saved best model for fold 2


Epoch 5: Loss = 0.0655, Val F1 = 0.9870
--> Saved best model for fold 2


Epoch 6: Loss = 0.0503, Val F1 = 0.9880
--> Saved best model for fold 2


Epoch 7: Loss = 0.0349, Val F1 = 0.9889
--> Saved best model for fold 2


KeyboardInterrupt: 

In [6]:
def run_inference():
    test_dataset = StoneDataset(root=CFG['data_root'], split='test', transforms=eval_transform)
    test_loader = DataLoader(test_dataset, batch_size=CFG['batch_size'], shuffle=False, 
                             num_workers=CFG['num_workers'], pin_memory=True)
                             
    # Container for all folds' predictions
    ensemble_preds = np.zeros(len(test_dataset), dtype=np.float32)
    
    for fold in range(CFG['n_splits']):
        model_path = f"weights/fold{fold}_{CFG['model_name']}_best.pt"
        if not os.path.exists(model_path):
            print(f"Skipping {model_path} - not found.")
            continue
            
        model = build_model(CFG['model_name']).to(CFG['device'])
        model.load_state_dict(torch.load(model_path, map_location=CFG['device'], weights_only=True))
        model.eval()
        
        fold_probs = []
        with torch.no_grad():
            for images, _ in tqdm(test_loader, desc=f"Inference Fold {fold}"):
                images = images.to(CFG['device'])
                probs = tta_forward(model, images, views=CFG['tta_views'])
                # Get probability for class 1
                fold_probs.extend(probs[:, 1].cpu().numpy())
                
        ensemble_preds += np.array(fold_probs) / CFG['n_splits']
        
    # Generate final predictions
    final_pred_classes = (ensemble_preds >= 0.5).astype(int)
    
    # Save to submission
    _, template_csv = resolve_test_paths(CFG['data_root'])
    submission_df = pd.read_csv(template_csv)
    # Ensure order matches, in StoneDataset it's sorted or reads as is, assuming index maps to id
    # Ensure you are assigning the correct labels to the correct IDs
    # (Assuming the test dataset returns samples in the same order as in submission.csv)
    
    submission_df['label'] = final_pred_classes
    submission_df.to_csv('submission_test.csv', index=False)
    print("Saved ensemble predictions to submission_test.csv")
    
test_preds = run_inference()

Loading local weights from ./weights/model.safetensors
Local weights load stats: {'loaded_tensors': 780, 'skipped_tensors': 2, 'missing_keys': 2, 'unexpected_keys': 0}


Inference Fold 0: 100%|██████████| 16/16 [00:03<00:00,  4.91it/s]


Loading local weights from ./weights/model.safetensors
Local weights load stats: {'loaded_tensors': 780, 'skipped_tensors': 2, 'missing_keys': 2, 'unexpected_keys': 0}


Inference Fold 1: 100%|██████████| 16/16 [00:02<00:00,  5.35it/s]


Loading local weights from ./weights/model.safetensors
Local weights load stats: {'loaded_tensors': 780, 'skipped_tensors': 2, 'missing_keys': 2, 'unexpected_keys': 0}


Inference Fold 2: 100%|██████████| 16/16 [00:02<00:00,  5.53it/s]


Skipping weights/fold3_tf_efficientnetv2_s_best.pt - not found.
Skipping weights/fold4_tf_efficientnetv2_s_best.pt - not found.
Saved ensemble predictions to submission_test.csv
